In [2]:
import numpy as np

def calculate_fitness(allocation, total_workload, server_capacities):
    """
    Evaluates fitness based on CPU utilization, memory usage, and response time.
    Lower score = better (more balanced and less overloaded).
    """

    allocation = np.maximum(allocation, 0)
    if np.sum(allocation) == 0:
        allocation = np.ones_like(allocation)
    allocation = (allocation / np.sum(allocation)) * total_workload


    cpu_util = allocation / server_capacities
    mem_util = (allocation * 0.85) / server_capacities


    response_time = np.where(cpu_util < 1.0, 1.0 / (1.05 - cpu_util), 50.0 * cpu_util)


    imbalance_penalty = np.var(cpu_util) * 100


    fitness = np.mean(cpu_util + mem_util) * 10 + np.mean(response_time) + imbalance_penalty
    return fitness, allocation

def get_best_neighbor(grid, fitness_grid, r, c, rows, cols):
    """Finds the best neighbor using a 4-neighbor (von Neumann) structure with wrap-around."""
    neighbors = [
        ((r - 1) % rows, c),  # Up
        ((r + 1) % rows, c),  # Down
        (r, (c - 1) % cols),  # Left
        (r, (c + 1) % cols)   # Right
    ]
    best_r, best_c = r, c
    best_fit = fitness_grid[r, c]

    for nr, nc in neighbors:
        if fitness_grid[nr, nc] < best_fit:
            best_fit = fitness_grid[nr, nc]
            best_r, best_c = nr, nc

    return grid[best_r, best_c]

def run_pca():
    print("--- Parallel Cellular Algorithm: Data Center Load Balancer ---")


    num_servers = int(input("Enter number of servers: "))
    capacities_input = input(f"Enter capacity for each of the {num_servers} servers (space-separated): ")
    server_capacities = np.array([float(x) for x in capacities_input.split()])

    total_workload = float(input("Enter total incoming workload to distribute: "))
    grid_size = int(input("Enter grid dimension (e.g., 4 for a 4x4 grid of cells): "))
    iterations = int(input("Enter number of iterations: "))
    alpha = float(input("Enter learning rate alpha (e.g., 0.5): "))


    grid = np.random.uniform(1, 100, size=(grid_size, grid_size, num_servers))
    fitness_grid = np.zeros((grid_size, grid_size))

    best_overall_fitness = float('inf')
    best_overall_allocation = None


    for t in range(iterations):

        for r in range(grid_size):
            for c in range(grid_size):
                fit, normalized_alloc = calculate_fitness(grid[r, c], total_workload, server_capacities)
                grid[r, c] = normalized_alloc
                fitness_grid[r, c] = fit

                if fit < best_overall_fitness:
                    best_overall_fitness = fit
                    best_overall_allocation = normalized_alloc.copy()


        new_grid = np.zeros_like(grid)
        for r in range(grid_size):
            for c in range(grid_size):
                x_best_neighbor = get_best_neighbor(grid, fitness_grid, r, c, grid_size, grid_size)

                new_grid[r, c] = grid[r, c] + alpha * (x_best_neighbor - grid[r, c])

        grid = new_grid


    print("\n--- Optimization Complete ---")
    print(f"Best Fitness Score (Lower is better): {best_overall_fitness:.4f}")
    print("\nOptimal Server Workload Distribution:")
    for i in range(num_servers):
        util_percent = (best_overall_allocation[i] / server_capacities[i]) * 100
        print(f"Server {i+1}: Assigned Load = {best_overall_allocation[i]:.2f} / {server_capacities[i]:.2f} "
              f"(Utilization: {util_percent:.1f}%)")

if __name__ == "__main__":
    run_pca()


--- Parallel Cellular Algorithm: Data Center Load Balancer ---
Enter number of servers: 4
Enter capacity for each of the 4 servers (space-separated): 100 150 80 120
Enter total incoming workload to distribute: 300
Enter grid dimension (e.g., 4 for a 4x4 grid of cells): 5
Enter number of iterations: 50
Enter learning rate alpha (e.g., 0.5): 0.6

--- Optimization Complete ---
Best Fitness Score (Lower is better): 15.0685

Optimal Server Workload Distribution:
Server 1: Assigned Load = 65.31 / 100.00 (Utilization: 65.3%)
Server 2: Assigned Load = 96.30 / 150.00 (Utilization: 64.2%)
Server 3: Assigned Load = 54.96 / 80.00 (Utilization: 68.7%)
Server 4: Assigned Load = 83.42 / 120.00 (Utilization: 69.5%)
